In [1]:
import openai

from typing import List, Iterator
import pandas as pd
import numpy as np
import os
import wget
import pickle
from ast import literal_eval

# Redis client library for Python
import redis

# I've set this to our new embeddings model, this can be changed to the embedding model of your choice
EMBEDDING_MODEL = "text-embedding-3-small"

# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

#### Load Data

In [2]:
transcript_df = pd.read_csv('data/embeddings.csv')

In [3]:
transcript_df.head()

,vector_id,content_vector,title_vector,video_id,chunk_id
0,0,"[0.010170252993702888, -0.015520867891609669, ...","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,0
1,1,"[0.00484744505956769, 0.008455636911094189, -0...","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,1
2,2,"[0.017064958810806274, -0.0008320452179759741,...","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,2
3,3,"[0.018084809184074402, -0.010019644163548946, ...","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,3
4,4,"[0.046452846378088, -0.005249288398772478, -0....","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,4


In [4]:
# Load title dictionary from pickle file
with open('data/title_dict.pkl', 'rb') as f:
    title_dict = pickle.load(f)

title_df = pd.DataFrame(title_dict.items(), columns=['video_id', 'title'])
title_df.head()

,video_id,title
0,q-wRvsiGYIs,"AMA #19: Collagen vs. Whey Protein, Creatine, ..."
1,ssmwxKPFMFU,Protocols to Improve Vision & Eyesight | Huber...
2,J7yn4tJEmJU,Tools for Overcoming Substance & Behavioral Ad...
3,7MEhDlw1e9k,How to Build Endurance | Huberman Lab Essentials
4,UyneMnERmnI,How to Improve Your Vitality & Heal From Disea...


In [5]:
# Join transcript_df with title_df on video_id to add titles
transcript_df = transcript_df.merge(title_df, on='video_id', how='left')
transcript_df.head()

,vector_id,content_vector,title_vector,video_id,chunk_id,title
0,0,"[0.010170252993702888, -0.015520867891609669, ...","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,0,Nutrients For Brain Health & Performance | Hub...
1,1,"[0.00484744505956769, 0.008455636911094189, -0...","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,1,Nutrients For Brain Health & Performance | Hub...
2,2,"[0.017064958810806274, -0.0008320452179759741,...","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,2,Nutrients For Brain Health & Performance | Hub...
3,3,"[0.018084809184074402, -0.010019644163548946, ...","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,3,Nutrients For Brain Health & Performance | Hub...
4,4,"[0.046452846378088, -0.005249288398772478, -0....","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,4,Nutrients For Brain Health & Performance | Hub...


In [6]:
# Read vectors from strings back into a list
transcript_df['title_vector'] = transcript_df.title_vector.apply(literal_eval)
transcript_df['content_vector'] = transcript_df.content_vector.apply(literal_eval)

# Set vector_id to be a string
transcript_df['vector_id'] = transcript_df['vector_id'].apply(str)

In [8]:
transcript_df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   vector_id       61 non-null     object
 1   content_vector  61 non-null     object
 2   title_vector    61 non-null     object
 3   video_id        61 non-null     object
 4   chunk_id        61 non-null     int64 
 5   title           61 non-null     object
dtypes: int64(1), object(5)
memory usage: 3.0+ KB


### Redis

#### Setup

In [ ]:
import redis
from redis.commands.search.indexDefinition import (
    IndexDefinition,
    IndexType
)
from redis.commands.search.query import Query
from redis.commands.search.field import (
    TextField,
    VectorField
)

REDIS_HOST =  "localhost"
REDIS_PORT = 6379
REDIS_PASSWORD = "" # default for passwordless Redis

# Connect to Redis
redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD
)
redis_client.ping()

True